In [4]:
from pickletools import optimize

import torch
from sympy import true
from torch import nn
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data import  Dataset
import torch.nn.functional as F

In [5]:
import torch.nn.functional as F

df = pd.read_csv("./data/rps.csv")


class RPSDataset(Dataset):
    def __init__(self, sequence_length):
        self.csv_path = "./data/rps.csv"
        self.sequence_length = sequence_length
        x = df.iloc[:,[-4,-3]].values
        moves = torch.tensor(x, dtype=torch.long)                       # (N, 2)
        self.data = F.one_hot(moves, num_classes=3).reshape(len(moves), -1).float()  # (N, 6)
        y = df.iloc[:,-3].values
        self.target = torch.tensor(y, dtype=torch.long)                 # stays 0/1/2
    def __len__(self):
        return len(self.data) - self.sequence_length
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.sequence_length]
        y = self.target[idx + self.sequence_length]
        return x, y

dataset = RPSDataset(sequence_length=5)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [7]:
class RPSNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)
    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        return self.fc(out[:,-1])

model = RPSNN(input_size=6,hidden_size=64,num_layers=2).to(device)


In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
def train(dataloader, model, loss_fn, optimizer):
    size = len(train_loader)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")





In [9]:
epochs = 500
print(len(dataset))
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
print("Done!")
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

97892
Epoch 1
-------------------------------
loss: 1.104335  [   64/ 1530]
loss: 1.096735  [ 6464/ 1530]
loss: 1.098675  [12864/ 1530]
loss: 1.099949  [19264/ 1530]
loss: 1.095432  [25664/ 1530]
loss: 1.094289  [32064/ 1530]
loss: 1.101559  [38464/ 1530]
loss: 1.096321  [44864/ 1530]
loss: 1.098827  [51264/ 1530]
loss: 1.088289  [57664/ 1530]
loss: 1.100644  [64064/ 1530]
loss: 1.098134  [70464/ 1530]
loss: 1.097202  [76864/ 1530]
loss: 1.097710  [83264/ 1530]
loss: 1.097324  [89664/ 1530]
loss: 1.097044  [96064/ 1530]
Epoch 2
-------------------------------
loss: 1.100332  [   64/ 1530]
loss: 1.099637  [ 6464/ 1530]
loss: 1.100727  [12864/ 1530]
loss: 1.096300  [19264/ 1530]
loss: 1.095909  [25664/ 1530]
loss: 1.100551  [32064/ 1530]
loss: 1.100040  [38464/ 1530]
loss: 1.102361  [44864/ 1530]
loss: 1.099332  [51264/ 1530]
loss: 1.097523  [57664/ 1530]
loss: 1.095905  [64064/ 1530]
loss: 1.098850  [70464/ 1530]
loss: 1.097143  [76864/ 1530]
loss: 1.098091  [83264/ 1530]
loss: 1.097711